# Train NOSHX, NOSHASHI's own model

This notebook trains **NOSHX** on NOSHASHI's own content and exports it for Ollama, free, on Google Colab's T4 GPU.

- **Base (default):** IBM Granite 4.0 1B, **Apache 2.0 licence**, so the trained model can be named, shipped and sold as NOSHX. It is small enough to answer quickly on an 8 GB laptop's CPU, beside NOSHASHI.
- **Base (optional):** Microsoft Phi-4-mini-instruct (3.8B, MIT licence). It writes better but is about two and a half times the size and slower. Set `BASE = "phi4-mini"` in the first code cell after the install to use it.
- **Data:** `scripts/noshx-model/noshx-train.jsonl` from the NOSHASHI repository: the in-app help, the Learn course's knowledge check and word list, the plan catalogue, the screen reference and every section of the published pages. Nothing in it is invented.
- **Result:** `noshx-q4_k_m.gguf` (about 1 GB from Granite, about 2.5 GB from Phi-4-mini) and the `Modelfile` that names it `noshx`.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**. Training Granite takes roughly 10–20 minutes (Phi-4-mini 20–40); the export adds about 10.


In [ ]:
%%capture
# Unsloth: fine-tuning that fits a free T4. Pulls matching torch, transformers and TRL.
!pip install unsloth

In [ ]:
# Which base to train. "granite" is the light default; "phi4-mini" is larger and slower.
BASE = "granite"

BASES = {
    # IBM Granite 4.0 1B, Apache 2.0. Turns: <|start_of_role|>user<|end_of_role|>…<|end_of_text|>
    "granite": dict(
        model="unsloth/granite-4.0-1b",
        load_in_4bit=False,  # small enough to train in 16-bit on a T4, which trains better
        modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
                 "shared_mlp.input_linear", "shared_mlp.output_linear"],
        user="<|start_of_role|>user<|end_of_role|>",
        assistant="<|start_of_role|>assistant<|end_of_role|>",
        modelfile="Modelfile",
        epochs=3,
    ),
    # Microsoft Phi-4-mini-instruct, MIT. Turns: <|user|>…<|end|><|assistant|>…<|end|>
    "phi4-mini": dict(
        model="unsloth/Phi-4-mini-instruct",
        load_in_4bit=True,  # fits the T4's 15 GB
        modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        user="<|user|>",
        assistant="<|assistant|>",
        modelfile="Modelfile.phi4-mini",
        epochs=2,
    ),
}
base = BASES[BASE]

# NOSHASHI's training data and the matching Modelfile, straight from the repository.
MODELFILE = base["modelfile"]
REPO = "https://raw.githubusercontent.com/Ignosha/noshashi/main/scripts/noshx-model"
!wget -q -O noshx-train.jsonl {REPO}/noshx-train.jsonl
!wget -q -O Modelfile {REPO}/{MODELFILE}
!wc -l noshx-train.jsonl


In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base["model"],
    max_seq_length=MAX_SEQ,
    load_in_4bit=base["load_in_4bit"],
)

# LoRA: train small adapters on every projection, not the whole model.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=base["modules"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="noshx-train.jsonl", split="train")

def to_text(example):
    # The base model's own chat template, so the trained model reads its turns the same way.
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

dataset = dataset.map(to_text)
print(dataset[0]["text"][:600])


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=base["epochs"],
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Learn to write NOSHX's answers, not to repeat the questions.
trainer = train_on_responses_only(trainer, instruction_part=base["user"], response_part=base["assistant"])

In [ ]:
stats = trainer.train()
print(stats)

In [ ]:
# A quick check before exporting: questions NOSHX should answer from what it learned.
FastLanguageModel.for_inference(model)
system = [m for m in dataset[0]["messages"] if m["role"] == "system"][0]["content"]
for question in [
    "How much does the Institutional plan cost?",
    "What is funded depth?",
    "Can you move my funds for me?",
]:
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
    print("Q:", question)
    print("A:", tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip(), "\n")

In [ ]:
# Export: merge the adapters and quantise to 4-bit GGUF for Ollama
# (about 1 GB from Granite, about 2.5 GB from Phi-4-mini).
import glob, os, shutil

model.save_pretrained_gguf("noshx_gguf", tokenizer, quantization_method="q4_k_m")

candidates = [p for p in glob.glob("**/*.gguf", recursive=True) if "q4_k_m" in p.lower()]
assert candidates, "No Q4_K_M GGUF was written; see the export log above."
shutil.move(candidates[0], "noshx-q4_k_m.gguf")
print("Wrote noshx-q4_k_m.gguf,", round(os.path.getsize("noshx-q4_k_m.gguf") / 1e9, 2), "GB")

In [ ]:
# Download both files (or copy them to Google Drive with the next cell).
from google.colab import files
files.download("Modelfile")
files.download("noshx-q4_k_m.gguf")

In [ ]:
# Optional: save to Google Drive instead of downloading through the browser.
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/noshx && cp noshx-q4_k_m.gguf Modelfile /content/drive/MyDrive/noshx/

## Add NOSHX to your laptop

1. Install Ollama from ollama.com/download.
2. Put `noshx-q4_k_m.gguf` and `Modelfile` in one folder, open a terminal there and run:

   ```
   ollama create noshx -f Modelfile
   ```
3. In NOSHASHI, open **NOSHX**, change the runtime to **Ollama**, and pick **noshx**.

The Modelfile matches the base you trained. NOSHX Core still reads the ledger and NOSHASHI's pages; the trained model phrases and reasons over what Core read. To retrain after the product changes, regenerate the data in the repository (`UPDATE_NOSHX_DATASET=1 npx vitest run src/lib/noshx/__tests__/dataset.test.ts`), push, and run this notebook again.